In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import gc

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 46.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.84MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.3MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.58MB/s]


In [3]:
class mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10)
        )

    def forward(self, x):
        return self.layers(x)

model = mlp()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())
gc.collect()

0

In [4]:
epochs = 10
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

In [5]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/10]')
    print("Training loss", (running_loss / total))  
    train_losses.append(running_loss / total)
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / total))
    test_losses.append(running_loss_eval / total)
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)
    
    gc.collect()

Epoch [1/10]
Training loss 0.1905517680366834
Training accuracy 94.16333333333333 %
Testing loss 0.14716450534760953
Testing accuracy 95.67 %
Epoch [2/10]
Training loss 0.08882820278642078
Training accuracy 97.24333333333334 %
Testing loss 0.08527223318815232
Testing accuracy 97.39 %
Epoch [3/10]
Training loss 0.06433663303783785
Training accuracy 98.01 %
Testing loss 0.08359577022492885
Testing accuracy 97.66 %
Epoch [4/10]
Training loss 0.051292724894778804
Training accuracy 98.38166666666666 %
Testing loss 0.07367979325354099
Testing accuracy 98.16 %
Epoch [5/10]
Training loss 0.04247477632336474
Training accuracy 98.705 %
Testing loss 0.08463889323174953
Testing accuracy 97.83 %
Epoch [6/10]
Training loss 0.03517987979699198
Training accuracy 98.855 %
Testing loss 0.07857378078624606
Testing accuracy 98.0 %
Epoch [7/10]
Training loss 0.03235346968949307
Training accuracy 98.96666666666667 %
Testing loss 0.07010079007595778
Testing accuracy 98.19 %
Epoch [8/10]
Training loss 0.03012